# **Prompting for Social Science Studies**

In this session, we will explore how to use LLMs for social science research.

In [ ]:
!pip install openai

## Relevancy Classification

Relevancy classification is a task that involves determining whether a given piece of text is relevant to a specific topic or question. This can be particularly useful where we often need to sift through large amounts of text data to find relevant information.

For this example, we will reuse the same `news` article from our previous session, and we will classify whether the the sentences in the article are relevant to the topic of "technology."

In [1]:
# excerpt from https://www.bbc.com/news/articles/czx499e6y9wo
news = """
A woodland classroom has been built at a popular nature reserve.
Northumberland Wildlife Trust has opened the facility at its site in Hauxley, near Morpeth, to provide an outdoor learning space for children and adults.
It replaces a playground, which the trust said had reached the end of its life.
The charity's Druridge Bay landscapes manager Alex Lister said wildlife-rich environments helped people's mental and physical health.
"People who have the opportunity to experience nature on their doorstep are more active and mentally resilient," he said.
The classroom is made using natural wood, including some from the trust's other reserves, and the organisation said it hoped it would complement the indoor learning space.
Baby and toddler groups, Scouts, Brownies, schools and creative groups are expected to make use of the space.
The classroom was funded through a £8,100 grant from the Nadara Sisters North Steads Wind Farm Community Benefit Fund.
"The new classroom will help more of the Hauxley community and beyond access nature in a new and different way," Lister added. 
"""

news_sentences = news.strip().split("\n")

We define the response schema that contain field relevancy that can be either "relevant" or "irrelevant".

In [ ]:
from openai import OpenAI
from pydantic import BaseModel
from typing import Optional, Literal

model = "liquid/lfm-2.5-1.2b-instruct:free"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="<OPENROUTER_API_KEY>",
)

# helper function to call the LLM
def completion(prompt: str, response: type[BaseModel]) -> Optional[BaseModel]:
    result = client.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        response_format=response,
    )

    return result.choices[0].message.parsed

To programmatically construct the prompt, we can use the templating capability of Python string. We will create a prompt with a placeholder for the text that we want to include in the prompt. We will then fill in the placeholder with the actual text that we want to classify.

In [ ]:
class RelevancyModel(BaseModel):
    relevancy: Literal["relevant", "irrelevant"]

prompt_relevancy = """
Classify whether this text is relevant or irrelevant to the topic of "technology".
The output should be either "relevant" or "irrelevant".

Text: {text}
"""

Then we will iterate through the sentences in the news article, and for each sentence, we will fill in the placeholder in the prompt with the sentence, and then we will send the prompt to the LLM to get the classification result.

In [ ]:
for sentence in news_sentences:
    filled_prompt = prompt_relevancy.format(text=sentence)
    result = completion(filled_prompt, RelevancyModel)
    print(f"Sentence: {sentence}")
    print(f"Relevancy: {result.relevancy}")
    print("-" * 50)

Sentence: A woodland classroom has been built at a popular nature reserve.
Relevancy: relevant
--------------------------------------------------
Sentence: Northumberland Wildlife Trust has opened the facility at its site in Hauxley, near Morpeth, to provide an outdoor learning space for children and adults.
Relevancy: relevant
--------------------------------------------------
Sentence: It replaces a playground, which the trust said had reached the end of its life.
Relevancy: relevant
--------------------------------------------------
Sentence: The charity's Druridge Bay landscapes manager Alex Lister said wildlife-rich environments helped people's mental and physical health.
Relevancy: relevant
--------------------------------------------------
Sentence: "People who have the opportunity to experience nature on their doorstep are more active and mentally resilient," he said.
Relevancy: irrelevant
--------------------------------------------------
Sentence: The classroom is made using 

Through the use schema validation for structured output, we can easily extract the relevancy classification result from the LLM's response and use it for further analysis or processing.

After that we can iterate the prompt for further improvements, for example if we find the LLM is often misclassifying sentences that contain certain keywords, we can modify the prompt to be more specific about how to handle those keywords, or we can provide additional context in the prompt to help the LLM make a more informed classification.

## Sentiment Analysis

Sentiment analysis is a task that involves determining the sentiment or emotion expressed in a piece of text. This can be useful for understanding the opinions towards a particular topic, product, or service.

Sentiment usually classified into categories such as "positive" and "negative". But depending on the use case, we can also include other categories such as "neutral" or "mixed", or even use a more fine-grained classification such as "very positive", "positive", "neutral", "negative", and "very negative".

For this example, we will classify the sentiment of the sentences in the same `news` article that we have been using in this session. We will classify whether the sentiment of each sentence is "positive", "negative", or "neutral" towards the topic of "technology".

In [6]:
class SentimentModel(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]

prompt_sentiment = """
Analyze the overall sentiment in the text towards the topic of "technology".
The sentiment can be classified as "positive", "negative", or "neutral".

Text: {text}
"""

In [8]:
for sentence in news_sentences:
    filled_prompt = prompt_sentiment.format(text=sentence)
    result = completion(filled_prompt, SentimentModel)
    print(f"Sentence: {sentence}")
    print(f"Relevancy: {result.sentiment}")
    print("-" * 50)

Sentence: A woodland classroom has been built at a popular nature reserve.
Relevancy: positive
--------------------------------------------------


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1776211200000'}, 'provider_name': None}}, 'user_id': 'user_31oK6UkrLE8YlSoP09ElppVYUug'}

We can utilize LLMs for various tasks in social science research. The key is to design effective prompts that can guide the LLM to produce the desired output, and to use structured output validation to ensure that we can easily extract and utilize the information we need from the LLM's responses.

Providing the correct prompt for the correct task is crucial for getting accurate and useful results from the LLM. It may take some experimentation and iteration to find the best prompt for a particular task, but with iteration and refinement, we can achieve good results for research tasks at hand.